In [1]:
import os
import math
from functools import reduce
from operator import add
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import NumericType
from pyspark.sql.functions import (
    col, when, count,
    to_date, dayofmonth, month, year, date_format, to_timestamp,
    hour as spark_hour, 
    sin, cos, last, first
)
spark = SparkSession.builder \
    .appName("Weather_Preprocessing_Jupyter") \
    .master("local[*]") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/20 16:52:26 WARN Utils: Your hostname, admin2025RESNAN, resolves to a loopback address: 127.0.1.1; using 172.27.14.10 instead (on interface eth0)
26/03/20 16:52:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 16:52:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
def step_1_drop_cols(df: DataFrame) -> DataFrame:
    # 1. Dổi tên cột
    new_columns = [
        'index', 'date', 'hour', 'prcp', 'askq', 'askkmax', 'askkmin', 'bxmt', 
        'tempkk', 'tempds', 'tmax', 'tmin', 'tdsmax', 'tdsmin', 'ttdmax', 'ttdmin', 
        'humi', 'windydirect', 'windygust', 'windyspeed', 'region', 'state', 'station', 
        'station_code', 'latitude', 'longitude', 'elevation'
    ]
    
    if len(df.columns) == len(new_columns):
        df = df.toDF(*new_columns)
    else:
        print(f"CẢNH BÁO: Số lượng cột thực tế ({len(df.columns)}) không khớp ({len(new_columns)}).")
    # 2. Bỏ cột không sài
    cols_to_drop = ['index', 'region', 'state', 'station']
    df = df.drop(*cols_to_drop)
    # 3. Định dạng lại cột hour (2026-03-16 00:00:00 -> 00:00:00)
    df = df.withColumn('hour', date_format(to_timestamp(col('hour')), 'HH:mm:ss'))

    # 4. Xử lý cột date và lọc năm
    if 'date' in df.columns:
        # Ép kiểu chuỗi về dạng chuẩn DateType của Spark
        df = df.withColumn('date', to_date(col('date')))
        
        # Tách thành các cột riêng biệt để dùng cho các bước sau
        df = df.withColumn('day', dayofmonth(col('date')))
        df = df.withColumn('month', month(col('date')))
        df = df.withColumn('year', year(col('date')))
        
        # # Lọc cắt bỏ toàn bộ dữ liệu trước năm 2015
        # df = df.filter(col('year') >= 2017)
        
    return df


def step_2_handle_missing(df):
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    all_cols = df.columns
    
    # 1. Chuyển các dữ liệu lỗi thành Null
    replace_exprs = [
        when(col(c) == -9999, None).otherwise(col(c)).alias(c) if c in numeric_cols else col(c) 
        for c in all_cols
    ]
    df = df.select(*replace_exprs)

    # 2. Xóa những ngày dữ liệu toàn lỗi
    cols_to_check = [
        'prcp', 'askq', 'askkmax', 'askkmin', 'bxmt', 
        'tempkk', 'tempds', 'tmax', 'tmin', 'tdsmax', 
        'tdsmin', 'ttdmax', 'ttdmin', 'humi', 'windydirect', 
        'windygust', 'windyspeed'
    ]
    
    # Chỉ xét những cột có thật trong df
    actual_cols = [c for c in cols_to_check if c in df.columns]

    if actual_cols and "station_code" in df.columns and "date" in df.columns:
        # Đếm số lượng Not Null của từng cột (Gom nhóm theo TRẠM và NGÀY)
        agg_exprs = [count(col(c)).alias(f"cnt_{c}") for c in actual_cols]
        daily_stats = df.groupBy("station_code", "date").agg(*agg_exprs)

        # Cộng dồn tất cả các biến count lại (Nếu = 0 nghĩa là trọn 24h của ngày đó đều NULL)
        total_valid_expr = reduce(add, [col(f"cnt_{c}") for c in actual_cols])

        # Lọc ra danh sách các NGÀY có ít nhất 1 dòng dữ liệu hợp lệ (> 0)
        valid_days = daily_stats.filter(total_valid_expr > 0).select("station_code", "date")

        # Inner Join: Những ngày toàn NULL sẽ bị loại bỏ sạch sẽ!
        df = df.join(valid_days, on=["station_code", "date"], how="inner")

    # 3. Điền dữ liệu khuyết bằng dữ liệu gần nhất
    # Sắp xếp theo ngày 
    window_ffill = Window.partitionBy("station_code").orderBy("date", "hour") \
                         .rowsBetween(Window.unboundedPreceding, Window.currentRow)
                         
    window_bfill = Window.partitionBy("station_code").orderBy("date", "hour") \
                         .rowsBetween(Window.currentRow, Window.unboundedFollowing)
    
    # Kéo giá trị hợp lệ từ quá khứ xuống
    ffill_exprs = [
        last(col(c), ignorenulls=True).over(window_ffill).alias(c) if c in numeric_cols else col(c) 
        for c in all_cols
    ]
    df = df.select(*ffill_exprs)

    # Đẩy giá trị hợp lệ từ tương lai lên nếu không có giá trị ngày quá khứ
    bfill_exprs = [
        first(col(c), ignorenulls=True).over(window_bfill).alias(c) if c in numeric_cols else col(c) 
        for c in all_cols
    ]
    df = df.select(*bfill_exprs)

    return df

def step_3_rain_label(df: DataFrame) -> DataFrame:
    # Gán nhãn
    if 'prcp' in df.columns:
        df = df.withColumn('rain_label', when(col('prcp') > 0, 1).otherwise(0))
    return df

def step_4_datetime_features(df: DataFrame) -> DataFrame:
    # 1. Tính Sin/Cos cho Tháng (Chu kỳ 12 tháng)
    if 'month' in df.columns:
        # Công thức: sin(month * 2 * pi / 12)
        df = df.withColumn('month_sin', sin(col('month') * (2 * math.pi / 12)))
        df = df.withColumn('month_cos', cos(col('month') * (2 * math.pi / 12)))

    # 2. Tính Sin/Cos cho Giờ (Chu kỳ 24 giờ)
    if 'hour' in df.columns:
        # Lấy số giờ từ định dạng HH:mm:ss đã làm ở Bước 1
        df = df.withColumn('_temp_hour', spark_hour(col('hour')).cast('double'))
        
        df = df.withColumn('hour_sin', sin(col('_temp_hour') * (2 * math.pi / 24)))
        df = df.withColumn('hour_cos', cos(col('_temp_hour') * (2 * math.pi / 24)))
        
        # Xóa cột tạm
        df = df.drop('_temp_hour')

    # 3. Xóa cột date vì đã tách thành cột ngày tháng năm, xóa năm để tránh overfit, xóa giờ tháng vì đã có sin cos, xóa ngày vì ngày 1 hay ngày 31 không quan trọng bằng việc nó là mùa nào
    cols_to_drop_final = ['date', 'hour', 'year', 'day', 'month']

    df = df.drop(*cols_to_drop_final)

    return df

def run_preprocessing_1(df: DataFrame) -> DataFrame:
    print("-> Chạy Bước 1...")
    df = step_1_drop_cols(df)
    print(f"Tổng số dòng sau B1: {df.count()}")

    print("-> Chạy Bước 2...")
    df = step_2_handle_missing(df)
    print(f"Tổng số dòng sau B2: {df.count()}")

    print("-> Chạy Bước 3...")
    df = step_3_rain_label(df)

    print("-> Chạy Bước 4...")
    df = step_4_datetime_features(df)

    return df

In [3]:
input_file = 'data.parquet'

print(f"Đang đọc dữ liệu từ '{input_file}'...")
# Sử dụng spark.read.parquet để đọc file
df = spark.read.parquet(input_file)

print(f"Tổng số dòng ban đầu: {df.count()}")

# Kiểm tra lại schema để đảm bảo dữ liệu khớp
df.printSchema()

print("\n--- BẮT ĐẦU LUỒNG TIỀN XỬ LÝ BƯỚC 1 TỚI 4 ---")
df_processed = run_preprocessing_1(df)

file_name = "danh_sach_cot.txt"
# Mở file để ghi
with open(file_name, "w", encoding="utf-8") as f:
    f.write(f"--- TỔNG CỘNG: {len(df_processed.columns)} CỘT ---\n")
    for i, col_name in enumerate(df_processed.columns, 1):
        f.write(f"{i}. {col_name}\n")

print(f"Đã xuất xong! Bạn hãy tìm file '{file_name}' ở cùng thư mục với file Jupyter này để mở nhé.")

Đang đọc dữ liệu từ 'data.parquet'...


Tổng số dòng ban đầu: 16260936
root
 |-- index: long (nullable = true)
 |-- Data: string (nullable = true)
 |-- Hora: string (nullable = true)
 |-- PRECIPITAÇÃO TOTAL, HORÁRIO (mm): double (nullable = true)
 |-- PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB): long (nullable = true)
 |-- PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB): double (nullable = true)
 |-- RADIACAO GLOBAL (Kj/m²): long (nullable = true)
 |-- TEMPERATURA DO AR - BULBO SECO, HORARIA (°C): double (nullable = true)
 |-- TEMPERATURA DO PONTO DE ORVALHO (°C): double (nullable = true)
 |-- TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- UMIDADE REL. MAX. NA HORA ANT. (AUT) (%):

Tổng số dòng sau B2: 14979072
-> Chạy Bước 3...
-> Chạy Bước 4...
Đã xuất xong! Bạn hãy tìm file 'danh_sach_cot.txt' ở cùng thư mục với file Jupyter này để mở nhé.
